# Battery State-of-Health (SoH) Estimation
NASA Battery Dataset — Kaggle cleaned version

Author: Wasik Billah Ibn Rashid
Purpose: Benchmark ensemble ML methods for SoH estimation
         and connect results to physical degradation mechanisms


In [ ]:
# Mount Drive

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Install xgboost

!pip install xgboost -q

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')



In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 —
# ─────────────────────────────────────────────────────────────────
BASE_PATH = r"/content/drive/MyDrive/Research/Kaggle_Battery_Dataset/archive/cleaned_dataset"
DATA_PATH = os.path.join(BASE_PATH, "data")
META_PATH = os.path.join(BASE_PATH, "metadata.csv")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — LOAD METADATA
# ─────────────────────────────────────────────────────────────────
print("Loading metadata...")
meta = pd.read_csv(META_PATH)

# Keep only discharge cycles (these have capacity measurements)
discharge = meta[meta['type'] == 'discharge'].copy() # Filter for discharge cycles and create a copy
discharge = discharge.dropna(subset=['Capacity']) # Remove rows where 'Capacity' is missing
discharge['Capacity'] = pd.to_numeric(discharge['Capacity'], errors='coerce') # Convert 'Capacity' to numeric, coercing errors to NaN
discharge = discharge.dropna(subset=['Capacity']) # Remove rows where 'Capacity' conversion resulted in NaN
discharge = discharge.reset_index(drop=True) # Reset the DataFrame index

# Show which batteries are available
print(f"Total discharge cycles: {len(discharge)}") # Print the total number of discharge cycles
print(f"Batteries in dataset:   {sorted(discharge['battery_id'].unique())}") # Print the unique battery IDs in sorted order
print()

# See the discharge dataframe
from google.colab import data_table
data_table.DataTable(discharge)

#start_time format [Year, Month, Day, Hour, Minute, Second]

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — COMPUTE SoH FOR EACH BATTERY
# SoH = capacity / initial capacity (first cycle of each battery)
# ─────────────────────────────────────────────────────────────────
initial_capacities = discharge.groupby('battery_id')['Capacity'].first() # Get the initial capacity for each battery
discharge['initial_capacity'] = discharge['battery_id'].map(initial_capacities) # Map initial capacities back to the discharge DataFrame
discharge['SoH'] = discharge['Capacity'] / discharge['initial_capacity'] # Calculate State-of-Health (SoH)

# Remove physically invalid SoH values caused by anomalous capacity readings
discharge = discharge[(discharge['SoH'] > 0.5) & (discharge['SoH'] <= 1.1)]
discharge = discharge.reset_index(drop=True)

# Add cycle number per battery
discharge['cycle_number'] = discharge.groupby('battery_id').cumcount() + 1 # Assign a sequential cycle number for each battery

data_table.DataTable(discharge)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — EXTRACT FEATURES FROM EACH DISCHARGE CYCLE

selected_batteries = ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0049', 'B0050', 'B0051', 'B0052', 'B0053', 'B0054', 'B0055', 'B0056']
discharge = discharge[discharge['battery_id'].isin(selected_batteries)]
discharge = discharge.reset_index(drop=True)

print(f"Using batteries: {selected_batteries}")
print(f"Total discharge cycles to process: {len(discharge)}")
print("Extracting features...\n")

print("Extracting features from discharge cycles...") # Inform the user about feature extraction
print("This may take a few minutes...\n") # Indicate that the process may take some time

features_list = [] # Initialize an empty list to store extracted features

for idx, row in discharge.iterrows(): # Iterate through each discharge cycle in the 'discharge' DataFrame
    filepath = os.path.join(DATA_PATH, row['filename']) # Construct the full file path for the cycle data

    try: # Start a try block to handle potential file reading errors
        cycle_data = pd.read_csv(filepath) # Read the CSV data for the current cycle

        # Skip cycles with insufficient data points
        if len(cycle_data) < 10: # Check if the cycle data has fewer than 10 rows
            continue # Skip to the next iteration if data points are insufficient

        v = cycle_data['Voltage_measured'].values # Extract voltage measurements
        t = cycle_data['Temperature_measured'].values # Extract temperature measurements
        time = cycle_data['Time'].values # Extract time measurements

        discharge_time  = time[-1] - time[0] # Calculate total discharge time
        max_temperature = np.max(t) # Calculate maximum temperature during discharge
        min_voltage     = np.min(v) # Calculate minimum voltage during discharge
        mean_voltage    = np.mean(v) # Calculate mean voltage during discharge
        voltage_drop    = v[0] - v[-1] # Calculate voltage drop from start to end

        features_list.append({ # Append the extracted features to the list
            'battery_id':       row['battery_id'], # Battery ID
            'cycle_number':     row['cycle_number'], # Cycle number
            'discharge_time':   discharge_time, # Calculated discharge time
            'max_temperature':  max_temperature, # Calculated maximum temperature
            'min_voltage':      min_voltage, # Calculated minimum voltage
            'mean_voltage':     mean_voltage, # Calculated mean voltage
            'voltage_drop':     voltage_drop, # Calculated voltage drop
            'SoH':              row['SoH'] # State-of-Health from the discharge DataFrame
        })

    except Exception as e: # Catch any exceptions during file processing
        continue # Continue to the next cycle if an error occurs

df = pd.DataFrame(features_list) # Create a Pandas DataFrame from the collected features

print(f"Feature extraction complete. Total cycles with features: {len(df)}") # Print the number of cycles with extracted features

print(f"\nFeature preview:") # Print a header for feature preview
print(df.head()) # Display the first few rows of the features DataFrame
print(f"\nSoH range: {df['SoH'].min():.3f} — {df['SoH'].max():.3f}") # Print the range of SoH values
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — PREPARE TRAIN / TEST SPLIT
# Split by cycle number: first 70% of cycles = train, last 30% = test
# ─────────────────────────────────────────────────────────────────
feature_cols = ['discharge_time', 'max_temperature',
                'min_voltage', 'mean_voltage', 'voltage_drop'] # Define the list of feature columns

# Split per battery to avoid data leakage
train_list, test_list = [], [] # Initialize empty lists for training and testing dataframes
for battery in df['battery_id'].unique(): # Iterate through each unique battery ID
    b_data = df[df['battery_id'] == battery].sort_values('cycle_number') # Filter data for the current battery and sort by cycle number
    split_idx = int(len(b_data) * 0.7) # Calculate the index for the 70/30 split
    train_list.append(b_data.iloc[:split_idx]) # Add the first 70% of cycles to the training list
    test_list.append(b_data.iloc[split_idx:]) # Add the remaining 30% of cycles to the testing list

train_df = pd.concat(train_list).reset_index(drop=True) # Concatenate training dataframes and reset index
test_df  = pd.concat(test_list).reset_index(drop=True) # Concatenate testing dataframes and reset index

X_train = train_df[feature_cols].values # Extract feature values for the training set
y_train = train_df['SoH'].values # Extract target values (SoH) for the training set
X_test  = test_df[feature_cols].values # Extract feature values for the test set
y_test  = test_df['SoH'].values # Extract target values (SoH) for the test set

# Normalize features
scaler = MinMaxScaler() # Initialize a MinMaxScaler
X_train_scaled = scaler.fit_transform(X_train) # Fit scaler on training features and transform them
X_test_scaled  = scaler.transform(X_test) # Transform test features using the fitted scaler

print(f"Training set: {len(X_train)} cycles") # Print the number of cycles in the training set
print(f"Test set:     {len(X_test)} cycles") # Print the number of cycles in the test set
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — TRAIN FOUR MODELS
# ─────────────────────────────────────────────────────────────────
print("Training models...")

models = {
    'Linear Regression':  LinearRegression(), # Initialize a basic Linear Regression model
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42), # Initialize a Random Forest regressor with 100 trees
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42), # Initialize a Gradient Boosting regressor
    'XGBoost':            XGBRegressor(n_estimators=100, random_state=42, verbosity=0) # Initialize an XGBoost regressor with silenced output
}

results = {} # Initialize a dictionary to store RMSE scores for each model
predictions = {} # Initialize a dictionary to store predicted SoH values for plotting

for name, model in models.items(): # Loop through each model in the defined dictionary
    model.fit(X_train_scaled, y_train) # Train the current model on the scaled training data
    y_pred = model.predict(X_test_scaled) # Generate predictions for the scaled test dataset
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Calculate the Root Mean Squared Error (RMSE)
    results[name] = rmse # Store the calculated RMSE in the results dictionary
    predictions[name] = y_pred # Store the prediction values for later visualization
    print(f"  {name:<25} RMSE = {rmse:.4f}") # Print the formatted model name and its RMSE score

print() # Print a blank line for readability
best_model_name = min(results, key=results.get) # Identify the model with the lowest RMSE score
print(f"Best model: {best_model_name} (RMSE = {results[best_model_name]:.4f})") # Print the name and score of the top-performing model
print()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — PLOT RESULTS
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(19, 10)) # Create a 2x2 grid of subplots for the four models
fig.suptitle('SoH Estimation — Predicted vs Actual\nNASA Battery Dataset', # Set the main title for the entire figure
             fontsize=14, fontweight='bold', y=0.98) # Format the main title font and position

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336'] # Define a list of colors for each model's prediction line

for ax, (name, y_pred), color in zip(axes.flatten(), predictions.items(), colors): # Iterate through subplots, model names, predictions, and colors
    rmse = results[name] # Retrieve the RMSE for the current model

    # Sort by actual SoH for cleaner plot
    sort_idx = np.argsort(y_test) # Get indices that would sort the actual SoH values
    ax.plot(y_test[sort_idx], color='black', linewidth=1.5, # Plot the actual SoH values in black
            label='Actual SoH', zorder=3) # Add label and ensure actual values appear on top
    ax.plot(y_pred[sort_idx], color=color, linewidth=1.2, # Plot the predicted SoH values using the model's color
            alpha=0.8, label=f'Predicted SoH', linestyle='--') # Set transparency, label, and dashed line style
    ax.set_title(f'{name}\nRMSE = {rmse:.4f}', fontweight='bold') # Set the subplot title with model name and RMSE
    ax.set_xlabel('Test sample index (sorted by actual SoH)') # Set the label for the x-axis
    ax.set_ylabel('SoH') # Set the label for the y-axis
    ax.legend(fontsize=8) # Add a legend to the subplot with smaller font
    ax.set_ylim([0.5, 1.05]) # Fix the y-axis range to standardize comparison
    ax.grid(True, alpha=0.3) # Enable grid lines with low transparency

plt.tight_layout() # Adjust the layout to prevent overlapping elements
plt.savefig('soh_predictions.png', dpi=150, bbox_inches='tight') # Save the resulting plot as a high-resolution image
plt.show() # Display the plot in the notebook
print("Plot saved as: soh_predictions.png") # Confirm to the user that the file has been saved

In [ ]:
best_model = models[best_model_name] # Retrieve the best trained model from the models dictionary

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    # Linear Regression has no feature_importances_ — use absolute coefficients instead
    importances = np.abs(best_model.coef_)

fi_df = pd.DataFrame({ # Create a new DataFrame to organize feature names and their scores
    'Feature': feature_cols, # Assign the list of feature names to the Feature column
    'Importance': importances # Assign the importance scores to the Importance column
}).sort_values('Importance', ascending=True) # Sort the DataFrame by importance in ascending order for the horizontal bar chart

fig, ax = plt.subplots(figsize=(8, 4)) # Initialize a matplotlib figure and axis with specific dimensions
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], # Create a horizontal bar chart using features and their importance values
               color=['#1F4E79', '#2E75B6', '#5B9BD5', '#9DC3E6', '#BDD7EE']) # Apply a custom blue color palette to the bars
ax.set_xlabel('Feature Importance Score') # Set the label for the horizontal x-axis
ax.set_title(f'{best_model_name} Feature Importance\nfor Battery SoH Estimation', # Add a descriptive title to the plot
             fontweight='bold') # Format the title text to be bold
for bar, val in zip(bars, fi_df['Importance']): # Iterate through each bar and its corresponding importance value
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2, # Position the text label at the end of each bar
            f'{val:.3f}', va='center', fontsize=9) # Display the importance value rounded to three decimal places
ax.grid(True, alpha=0.3, axis='x') # Add vertical grid lines with low transparency for better readability
plt.tight_layout() # Automatically adjust the subplot parameters to give the plot more room
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight') # Save the plot as a high-resolution PNG file
plt.show() # Render and display the plot in the notebook
print("Feature importance plot saved as: feature_importance.png") # Print a confirmation message indicating the file was saved

In [ ]:
sample_battery = df['battery_id'].unique()[0] # Select the first unique battery ID from the dataset for visualization
b_data = df[df['battery_id'] == sample_battery].sort_values('cycle_number') # Filter data for the chosen battery and sort by cycle number

fig, ax = plt.subplots(figsize=(9, 4)) # Create a new figure and axis object with a specific width and height
ax.plot(b_data['cycle_number'], b_data['SoH'], # Plot cycle number against State-of-Health
        color='#1F4E79', linewidth=2, marker='o', markersize=3, label='Actual SoH') # Style the line with a dark blue color and markers
ax.axhline(y=0.8, color='red', linestyle='--', linewidth=1.5, # Add a horizontal dashed red line at SoH = 0.8
           label='End-of-life threshold (SoH = 0.80)') # Label the horizontal line as the EOL threshold
ax.set_xlabel('Cycle Number') # Set the label for the horizontal x-axis
ax.set_ylabel('State of Health (SoH)') # Set the label for the vertical y-axis
ax.set_title(f'Battery Degradation Curve — {sample_battery}', # Set the plot title including the specific battery ID
             fontweight='bold') # Make the title font weight bold
ax.legend() # Display the legend to identify the plotted lines
ax.grid(True, alpha=0.3) # Enable a light grid for easier data alignment
ax.set_ylim([0.5, 1.05]) # Fix the y-axis limits to focus on the relevant SoH range
plt.tight_layout() # Adjust layout to ensure labels and titles fit within the figure area
plt.savefig('degradation_curve.png', dpi=150, bbox_inches='tight') # Save the plot to a high-resolution PNG file
plt.show() # Render and display the plot in the notebook output
print("Degradation curve saved as: degradation_curve.png") # Print a confirmation message indicating the file was saved